<h1 align="center">Feature Selection</h1>

Inputs: the cleaned checkpoint from `data_cleaning.ipynb` (`dataset/cleaned_311.csv`).

In [10]:
import os
if not os.path.isdir('dataset') and os.path.isdir('../dataset'):
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

df = pd.read_csv('dataset/cleaned_311.csv', low_memory=False)
df['Created Date'] = pd.to_datetime(df['Created Date'])
print(df.shape)
df.head()

(3759919, 10)


,Created Date,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Agency,Borough,Location Type,Open Data Channel Type,Community Board,Police Precinct,resolution_hours
0,2026-09-20 01:39:36,noise - general,loud music/party,nypd,brooklyn,residential building,online,05 brooklyn,precinct 75,0.132222
1,2026-09-20 01:37:27,noise - general,loud music/party,nypd,brooklyn,street/sidewalk,mobile,01 brooklyn,precinct 94,0.166389
2,2026-09-20 01:36:25,noise - general,loud music/party,nypd,manhattan,park/playground,online,12 manhattan,precinct 33,0.149444
3,2026-09-20 01:35:40,noise - general,loud talking,nypd,brooklyn,street/sidewalk,mobile,08 brooklyn,precinct 77,0.230278
4,2026-09-20 01:34:30,noise - general,banging/pounding,nypd,manhattan,club/bar/restaurant,phone,03 manhattan,precinct 7,0.171944


### Time-based features

**`Created Date`** is the timestamp when the 311 request was filed (the start of the clock for resolution time). In the raw data it's text such as `09/20/2026 01:51:01 AM`; it's parsed to a datetime so pandas can extract parts of it. In the cleaned checkpoint it spans **2025-09-21 to 2026-09-20** with no missing values. It's the source for every time feature below, and the raw column itself is not used as a feature.

Derived features:
- `created_hour`: 1–24, with midnight as 24 (so 1 = 1 AM, 23 = 11 PM, 24 = midnight).
- `created_month`: 1–12.
- `day_of_week`: 1–7, where 1 = Monday and 7 = Sunday.
- `time_of_day`: night (0–5), morning (6–11), afternoon (12–17), evening (18–23).
- `is_weekend`: 1 for Saturday or Sunday.

Note: numeric hour treats 24 and 1 as far apart, even though they're neighbours on the clock. The time-of-day category avoids that issue, and we check both below.

In [11]:
h = df['Created Date'].dt.hour
df['created_hour'] = h.where(h > 0, 24)
df['created_month'] = df['Created Date'].dt.month
df['day_of_week'] = df['Created Date'].dt.dayofweek + 1
df['time_of_day'] = pd.cut(h, bins=[-1, 5, 11, 17, 23], labels=['night', 'morning', 'afternoon', 'evening'])
df['is_weekend'] = (df['day_of_week'] >= 6).astype(int)

df['log_resolution_hours'] = np.log1p(df['resolution_hours'])
df[['created_hour', 'created_month', 'day_of_week', 'time_of_day', 'is_weekend']].head()

,created_hour,created_month,day_of_week,time_of_day,is_weekend
0,1,9,7,night,1
1,1,9,7,night,1
2,1,9,7,night,1
3,1,9,7,night,1
4,1,9,7,night,1


### Categorical features association (Cramér's V)

Association between the categorical features, to spot overlap before choosing which to keep.

#### Why Cramér's V, and how it's calculated

**Why:** the categorical features have no numeric order (for example, "Agency = NYPD" is not "bigger than" "Agency = HPD"), so Pearson and Spearman correlation don't apply to them. Cramér's V works on counts in a contingency table instead. It runs from 0 (the two features are independent) to 1 (one fully determines the other), and it works for any number of categories. It's also symmetric, so the matrix is consistent.

**How it's calculated:**

1. **Build a contingency table** of counts for each pair of features (rows are categories of one feature, columns are categories of the other).

2. **Compute the chi-square statistic**, which compares observed counts $O$ with the counts $E$ expected if the two features were independent:

$$\chi^2 = \sum \frac{(O - E)^2}{E}, \qquad E = \frac{\text{row total} \times \text{column total}}{N}$$

3. **Scale it to 0–1:**

$$V = \sqrt{\frac{\chi^2 / N}{\min(r - 1,\ k - 1)}}$$

where $N$ is the total count, $r$ the number of rows, and $k$ the number of columns. Dividing by $\min(r-1, k-1)$ keeps the value between 0 and 1 regardless of table size.

**Note:** with about 3.7 million rows, the chi-square test is significant for almost any pair, so the p-value is not useful for deciding what to keep. The size of $V$ is what matters.

For resolution hours (a continuous target), the categorical-to-target column uses the correlation ratio $\eta$ instead, which measures how much of the variance in log resolution time is explained by the category.

In [12]:
from scipy.stats import chi2_contingency

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    return np.sqrt((chi2 / n) / min(r - 1, k - 1))

def correlation_ratio(x, y):
    g = y.groupby(x)
    ss_between = (g.size() * (g.mean() - y.mean()) ** 2).sum()
    return np.sqrt(ss_between / ((y - y.mean()) ** 2).sum())

cols = ['Problem (formerly Complaint Type)', 'Agency', 'Location Type', 'Open Data Channel Type', 'time_of_day']
log_target = np.log1p(df['resolution_hours'])

cramer = pd.DataFrame(
    [[cramers_v(df[a], df[b]) for b in cols] for a in cols],
    index=cols, columns=cols,
)
# resolution hours is continuous, so categorical-to-target uses the correlation ratio (on log hours)
cramer['Resolution hours (η)'] = [correlation_ratio(df[a], log_target) for a in cols]
cramer = cramer.round(2)
cramer

/tmp/ipykernel_38502/281183866.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y.groupby(x)


,Problem (formerly Complaint Type),Agency,Location Type,Open Data Channel Type,time_of_day,Resolution hours (η)
Problem (formerly Complaint Type),1.00,0.79,0.69,0.54,0.22,0.83
Agency,0.79,1.00,0.56,0.47,0.15,0.76
Location Type,0.69,0.56,1.00,0.42,0.13,0.56
Open Data Channel Type,0.54,0.47,0.42,1.00,0.09,0.36
time_of_day,0.22,0.15,0.13,0.09,1.00,0.20


***Findings:***
- **Complaint type overlaps strongly with agency (0.79) and location type (0.69).** Those three carry much of the same information, so the model doesn't need all of them at full weight.
- **Agency and location type also overlap (0.56).**
- **Open data channel is only moderately related to the others** (0.42 to 0.54), so it adds some independent information.
- **Time of day is almost independent of the other categorical features** (0.09 to 0.22). It's a useful, non-redundant addition.
- **Against resolution hours (η):** complaint type 0.83, agency 0.76, location type 0.56, open data channel 0.36, time of day 0.20. Complaint type and agency are the strongest predictors of resolution time.

### Correlation matrix (numerical features)

Spearman correlation between the numeric features and the target (`resolution_hours`). Spearman is used because the target is heavily skewed, and it measures rank-based association.

In [13]:
num_cols = ['created_hour', 'created_month', 'day_of_week', 'resolution_hours']
numeric_corr = df[num_cols].corr(method='spearman').round(3)
numeric_corr

,created_hour,created_month,day_of_week,resolution_hours
created_hour,1.000,0.001,0.042,-0.082
created_month,0.001,1.000,-0.000,-0.055
day_of_week,0.042,-0.000,1.000,-0.085
resolution_hours,-0.082,-0.055,-0.085,1.000


***Findings:***
- **The time features are almost uncorrelated with each other** (|r| ≤ 0.04), so they don't overlap. Each carries its own information.
- **Each time feature is weakly related to resolution time:** hour (-0.08), day of week (-0.09), and month (-0.06). Complaints filed later in the day or week resolve slightly faster, but the effect is small.
- **Pearson on raw hours gives smaller values** (about -0.02 to -0.04), because the skewed target pulls the linear correlation down. Spearman reflects the ranking more accurately, so it's the right measure here.

### Final feature set

Target: `resolution_hours`. Raw `Created Date` is not used as a feature (it's the source for the time features).

In [14]:
features = [
    'Problem (formerly Complaint Type)',
    'Agency',
    'Location Type',
    'Open Data Channel Type',
    'created_hour',
    'created_month',
    'day_of_week',
    'time_of_day',
]
target = 'resolution_hours'

model_df = df[features + [target]]
print(f"Features: {len(features)}  |  Rows: {len(model_df):,}")
print(model_df[features].nunique())

# Checkpoint for the modeling notebook. dataset/*.csv is git-ignored, so this stays local.
model_df.to_csv('dataset/model_features.csv', index=False)
print("\nSaved dataset/model_features.csv")

Features: 8  |  Rows: 3,759,919
Problem (formerly Complaint Type)    73
Agency                               16
Location Type                        22
Open Data Channel Type                5
created_hour                         24
created_month                        12
day_of_week                           7
time_of_day                           4
dtype: int64

Saved dataset/model_features.csv
